In [ ]:
import os, json, math, random
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier, VotingClassifier, StackingClassifier
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, precision_recall_curve, confusion_matrix, f1_score

import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# Reduce TF log noise
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

# Optional GPU memory growth
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for g in gpus:
            tf.config.experimental.set_memory_growth(g, True)
    except Exception as e:
        print("GPU memory growth warning:", e)

# Paths & constants
DATA_CSV      = "data_final.csv"
IMG_SIZE      = (224, 224)
BATCH_SIZE    = 64

OUT_DIR       = Path("outputs_logreg")
CACHE_DIR     = OUT_DIR / "cache"
ARTIFACTS_DIR = OUT_DIR / "artifacts"
for d in [OUT_DIR, CACHE_DIR, ARTIFACTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

EMB_NPY_PATH  = CACHE_DIR / "embeddings_resnet50.npy"
EMB_MAP_JSON  = CACHE_DIR / "emb_index.json"
SCALER_PATH   = ARTIFACTS_DIR / "scaler.pkl"
CFG_JSON      = ARTIFACTS_DIR / "feature_config.json"

BEST_MODEL_PATH = ARTIFACTS_DIR / "best_model.pkl"
BEST_THR_PATH   = ARTIFACTS_DIR / "best_threshold.json"


In [2]:
# %%
# === Load data ===
df = pd.read_csv(DATA_CSV)
required_cols = {"path_a","path_b","duplicate"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing columns {missing}. Need {required_cols}")

if df["duplicate"].dtype == object:
    le = LabelEncoder()
    df["duplicate"] = le.fit_transform(df["duplicate"])
    print("LabelEncoder for 'duplicate', classes:", list(le.classes_))

before = len(df)
df = df.dropna(subset=["path_a","path_b","duplicate"])
print(f"Dropped {before - len(df)} NA rows (if any)")

def file_exists(p): 
    try: return Path(p).is_file()
    except: return False
mask_exist = df["path_a"].apply(file_exists) & df["path_b"].apply(file_exists)
if not mask_exist.all():
    print(f"{(~mask_exist).sum()} rows with missing files -> removed")
    df = df[mask_exist].reset_index(drop=True)

print("Label distribution:\n", df["duplicate"].value_counts())
display(df.head())


Dropped 0 NA rows (if any)
Label distribution:
 duplicate
True     260
False    260
Name: count, dtype: int64


,path_a,title_a,path_b,title_b,duplicate
0,images_png/camera/194121937.png,Camera IP Wifi Ngoài Trời EZVIZ C3TN 3MP 2K Co...,images_png/camera/194130039.png,Camera IP Wifi Ngoài Trời EZVIZ H3 5MP Độ Phân...,True
1,images_png/phone/273990229.png,Điện Thoại Xiaomi Redmi Note 13 6GB/128GB - Hà...,images_png/phone/276944892.png,Điện Thoại Samsung Galaxy A06 4GB/64GB - Hàng ...,True
2,images_png/camera/275938579.png,"Camera trong nhà TP-Link Tapo TC60 - Full HD, ...",images_png/phone/277777809.png,"Điện thoại Samsung Galaxy A26 5G (8/128GB), Mặ...",False
3,images_png/phone/276274790.png,Điện Thoại Samsung Galaxy A06 4GB/128GB - Hàng...,images_png/phone/276944892.png,Điện Thoại Samsung Galaxy A06 4GB/64GB - Hàng ...,True
4,images_png/phone/276639232.png,Điện Thoại Samsung Galaxy A16 5G 8GB/128GB - H...,images_png/phone/276944868.png,Điện Thoại Samsung Galaxy A16 4GB/128GB - Hàng...,True


In [3]:
# %%
# === Split ===
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=SEED, stratify=df["duplicate"]
)
y_train = train_df["duplicate"].to_numpy()
y_test  = test_df["duplicate"].to_numpy()
print(f"Train: {len(train_df)} | Test: {len(test_df)}")


Train: 416 | Test: 104


In [4]:
# %%
# === ResNet50 embeddings (cached) ===
base = ResNet50(weights="imagenet", include_top=False, pooling="avg")
base.trainable = False

def tf_load(path_str):
    img = tf.io.read_file(path_str)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32)
    img = preprocess_input(img)
    return img

def embed_batch(paths):
    ds = tf.data.Dataset.from_tensor_slices([str(p) for p in paths])
    ds = ds.map(tf_load, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE)
    outs = []
    for batch in ds:
        emb = base(batch, training=False).numpy()
        outs.append(emb)
    return np.vstack(outs) if outs else np.zeros((0, 2048), dtype=np.float32)

def ensure_embeddings(all_paths):
    if EMB_NPY_PATH.exists() and EMB_MAP_JSON.exists():
        emb = np.load(EMB_NPY_PATH)
        with open(EMB_MAP_JSON, "r") as f:
            idx_map = json.load(f)
    else:
        emb = np.zeros((0, 2048), dtype=np.float32)
        idx_map = {}

    to_add = [p for p in all_paths if p not in idx_map]
    if to_add:
        print(f"Embedding {len(to_add)} new images (cache has {len(idx_map)}) ...")
        new_emb = embed_batch(to_add)
        start = len(emb)
        emb = np.vstack([emb, new_emb]) if len(emb) else new_emb
        for i, p in enumerate(to_add):
            idx_map[p] = start + i
        np.save(EMB_NPY_PATH, emb)
        with open(EMB_MAP_JSON, "w") as f:
            json.dump(idx_map, f)
        print("Updated cache:", EMB_NPY_PATH, EMB_MAP_JSON)
    else:
        print(f"Using cached embeddings ({len(idx_map)} images)")
    return emb, idx_map

unique_paths = pd.unique(pd.concat([train_df["path_a"], train_df["path_b"], test_df["path_a"], test_df["path_b"]])).astype(str)
emb_all, emb_index = ensure_embeddings(unique_paths)

def get_emb_of(paths):
    idxs = [emb_index[p] for p in paths]
    return emb_all[idxs]


Using cached embeddings (311 images)


In [5]:
# %%
# === Features ===
def pair_features(emb_a, emb_b):
    abs_diff = np.abs(emb_a - emb_b)
    prod     = emb_a * emb_b
    dot = np.sum(emb_a * emb_b, axis=1)
    na  = np.linalg.norm(emb_a, axis=1) + 1e-12
    nb  = np.linalg.norm(emb_b, axis=1) + 1e-12
    cos = (dot / (na * nb)).reshape(-1, 1)
    l2  = np.linalg.norm(emb_a - emb_b, axis=1).reshape(-1, 1)
    return np.hstack([abs_diff, prod, cos, l2])  # 4098 dims

emb_a_tr = get_emb_of(train_df["path_a"].astype(str).tolist())
emb_b_tr = get_emb_of(train_df["path_b"].astype(str).tolist())
X_train  = pair_features(emb_a_tr, emb_b_tr)

emb_a_te = get_emb_of(test_df["path_a"].astype(str).tolist())
emb_b_te = get_emb_of(test_df["path_b"].astype(str).tolist())
X_test   = pair_features(emb_a_te, emb_b_te)

print("Feature shapes:", X_train.shape, X_test.shape)


Feature shapes: (416, 4098) (104, 4098)


In [6]:
# %%
# === Scale (shared) ===
scaler = StandardScaler(with_mean=True, with_std=True)
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

joblib.dump(scaler, SCALER_PATH)
with open(CFG_JSON, "w") as f:
    json.dump({
        "img_size": [224, 224],
        "features": ["abs_diff(2048)", "prod(2048)", "cos(1)", "l2(1)"],
        "embedding": "ResNet50(ImageNet, avg pool, frozen)"
    }, f, indent=2)
print(f"Saved scaler → {SCALER_PATH}")


Saved scaler → outputs_logreg/artifacts/scaler.pkl


In [ ]:
# %%
# === Train models ===
models = {}

models["lr"] = LogisticRegression(
    max_iter=2000, class_weight="balanced", n_jobs=-1, solver="lbfgs"
)

models["rf"] = RandomForestClassifier(
    n_estimators=400, max_depth=None, min_samples_split=2,
    n_jobs=-1, class_weight="balanced_subsample", random_state=SEED
)

models["gb"] = GradientBoostingClassifier(
    learning_rate=0.05, n_estimators=400, max_depth=3, random_state=SEED
)

models["et"] = ExtraTreesClassifier(
    n_estimators=500, max_depth=None, n_jobs=-1,
    class_weight="balanced_subsample", random_state=SEED
)

base_svc = LinearSVC(C=1.0, class_weight="balanced", random_state=SEED)
models["svm_cal"] = CalibratedClassifierCV(estimator=base_svc, cv=3)

for name, clf in models.items():
    print(f"Training: {name}")
    clf.fit(X_train_sc, y_train)
    joblib.dump(clf, ARTIFACTS_DIR / f"{name}.pkl")
print("Saved base models.")


Training: lr
Training: rf
Training: gb


In [ ]:
# %%
# === Ensembles ===
lr  = models["lr"]; rf = models["rf"]; gb = models["gb"]; et = models["et"]; svm = models["svm_cal"]

voting_soft = VotingClassifier(
    estimators=[("lr", lr), ("rf", rf), ("gb", gb), ("et", et)],
    voting="soft", weights=[1.0, 1.2, 1.1, 1.1], n_jobs=-1
)
voting_soft.fit(X_train_sc, y_train)
joblib.dump(voting_soft, ARTIFACTS_DIR / "voting_soft.pkl")

stacking = StackingClassifier(
    estimators=[("lr", lr), ("rf", rf), ("gb", gb), ("et", et), ("svm", svm)],
    final_estimator=LogisticRegression(max_iter=2000, class_weight="balanced", solver="lbfgs"),
    stack_method="auto", n_jobs=-1, passthrough=False
)
stacking.fit(X_train_sc, y_train)
joblib.dump(stacking, ARTIFACTS_DIR / "stacking.pkl")

print("Ensembles trained & saved.")


In [ ]:
# %%
# === Evaluate & Auto-select best by F1 ===
def evaluate_model(name, clf, X_te, y_te, out_dir):
    if hasattr(clf, "predict_proba"):
        proba = clf.predict_proba(X_te)[:, 1]
    else:
        scores = clf.decision_function(X_te)
        smin, smax = scores.min(), scores.max()
        proba = (scores - smin) / (smax - smin + 1e-9)

    pred05 = (proba >= 0.5).astype(int)
    report = classification_report(y_te, pred05, digits=4, output_dict=True)
    auc = roc_auc_score(y_te, proba)
    cm = confusion_matrix(y_te, pred05)

    best_t, best_f1 = 0.5, f1_score(y_te, pred05)
    for t in np.linspace(0.05, 0.95, 19):
        f1 = f1_score(y_te, (proba >= t).astype(int))
        if f1 > best_f1:
            best_f1, best_t = f1, t

    with open(out_dir / f"{name}_metrics.json", "w") as f:
        json.dump({
            "roc_auc": float(auc),
            "confusion_matrix": cm.tolist(),
            "report": report,
            "best_threshold": float(best_t),
            "best_f1": float(best_f1)
        }, f, indent=2)

    with open(out_dir / f"{name}_threshold.json", "w") as f:
        json.dump({"threshold": float(best_t), "f1_on_test": float(best_f1)}, f, indent=2)

    print(f"[{name}] AUC={auc:.4f}  BestF1={best_f1:.4f} @thr={best_t:.2f}")
    return {"auc": auc, "best_f1": best_f1, "best_thr": best_t}

# Evaluate all models
all_models = {
    "lr": joblib.load(ARTIFACTS_DIR / "lr.pkl"),
    "rf": joblib.load(ARTIFACTS_DIR / "rf.pkl"),
    "gb": joblib.load(ARTIFACTS_DIR / "gb.pkl"),
    "et": joblib.load(ARTIFACTS_DIR / "et.pkl"),
    "svm_cal": joblib.load(ARTIFACTS_DIR / "svm_cal.pkl"),
    "voting_soft": joblib.load(ARTIFACTS_DIR / "voting_soft.pkl"),
    "stacking": joblib.load(ARTIFACTS_DIR / "stacking.pkl"),
}

results = {}
for name, clf in all_models.items():
    results[name] = evaluate_model(name, clf, X_test_sc, y_test, ARTIFACTS_DIR)

# Pick best by F1; tie-breaker = higher AUC; then simpler model name alphabetical
best_name = None
best_f1 = -1.0
best_auc = -1.0

for name, r in results.items():
    f1, auc = r["best_f1"], r["auc"]
    if (f1 > best_f1) or (math.isclose(f1, best_f1) and auc > best_auc) or (math.isclose(f1, best_f1) and math.isclose(auc, best_auc) and (best_name is None or name < best_name)):
        best_name, best_f1, best_auc = name, f1, auc

# Save BEST model and its threshold
joblib.dump(all_models[best_name], BEST_MODEL_PATH)

thr_file = ARTIFACTS_DIR / f"{best_name}_threshold.json"
if thr_file.exists():
    with open(thr_file, "r") as f:
        thr = float(json.load(f).get("threshold", 0.5))
else:
    thr = 0.5

with open(BEST_THR_PATH, "w") as f:
    json.dump({"model": best_name, "threshold": float(thr), "best_f1": float(best_f1), "auc": float(best_auc)}, f, indent=2)

print("\n=== BEST MODEL SELECTED ===")
print(f"Best by F1: {best_name}  F1={best_f1:.4f}  AUC={best_auc:.4f}  thr={thr:.2f}")
print(f"Saved best model → {BEST_MODEL_PATH}")
print(f"Saved best threshold → {BEST_THR_PATH}")


In [ ]:
# %%
# === Inference (defaults to BEST model) ===
def make_features_for_pair(path_a, path_b):
    ea = get_emb_of([path_a])
    eb = get_emb_of([path_b])
    return pair_features(ea, eb)

def predict_pair(path_a: str, path_b: str, model_name: str = None):
    scaler_ = joblib.load(SCALER_PATH)

    if model_name is None:
        # load best
        clf = joblib.load(BEST_MODEL_PATH)
        with open(BEST_THR_PATH, "r") as f:
            meta = json.load(f)
        thr = float(meta.get("threshold", 0.5))
        chosen = meta.get("model", "best_model")
    else:
        # load specific model
        clf = joblib.load(ARTIFACTS_DIR / f"{model_name}.pkl")
        thr_path = ARTIFACTS_DIR / f"{model_name}_threshold.json"
        if thr_path.exists():
            with open(thr_path, "r") as f:
                thr = float(json.load(f).get("threshold", 0.5))
        else:
            thr = 0.5
        chosen = model_name

    X  = make_features_for_pair(path_a, path_b)
    Xs = scaler_.transform(X)

    if hasattr(clf, "predict_proba"):
        p = float(clf.predict_proba(Xs)[0,1])
    else:
        s = float(clf.decision_function(Xs)[0])
        p = 1.0 / (1.0 + math.exp(-s))

    y = int(p >= thr)
    return chosen, y, p, thr

# Demo with a random test sample
row = test_df.sample(1, random_state=random.randint(0, 10**9)).iloc[0]
pa, pb, gt = str(row["path_a"]), str(row["path_b"]), int(row["duplicate"])
chosen, y, p, thr = predict_pair(pa, pb)  # default: best model

print("\nRandom Pair Prediction — Using BEST model")
print(f"Model: {chosen}")
print(f"Image A: {pa}")
print(f"Image B: {pb}")
print(f"GT: {gt} | Pred: {y} (p={p:.3f}, thr={thr:.2f})")
